# Pre-entrega 6 · Orquestador multi-agente (Código Tributario de Uruguay)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/matiasandrade7/llm-multi-agent-orchestrator/blob/main/notebooks/demo.ipynb)

Este notebook **demuestra el flujo de delegación**: un Supervisor decide en cada paso si interviene
el **Investigador** (RAG sobre el Código Tributario), el **Analista** (cálculos) o si ya se puede
responder.

Qué vas a ver:
1. Setup y topología del grafo (diagrama Mermaid).
2. Una consulta real que recorre **investigar → calcular → sintetizar**.
3. Una consulta que **solo investiga** (el analista no interviene).
4. Una consulta **fuera de tema** (el sistema declara que no hay evidencia, no inventa).
5. Cómo se aísla el contexto de cada agente.

> Requiere Python 3.12+. Si no tenés las API keys, ejecutá solo la sección **"Modo simulación"** al final.

## 1. Setup

In [ ]:
import os, pathlib, sys

REPO_URL = "https://github.com/matiasandrade7/llm-multi-agent-orchestrator.git"

# En Colab clonamos el repo; si el notebook corre dentro del repo (local), no hace falta.
if not pathlib.Path("src").exists():
    if not pathlib.Path("llm-multi-agent-orchestrator").exists():
        !git clone {REPO_URL}
    %cd llm-multi-agent-orchestrator

!pip install -q -r requirements.txt

In [ ]:
# Claves: Colab Secrets (icono de la llave) o variables de entorno / archivo .env en local.
try:
    from google.colab import userdata
    for name in ("GOOGLE_API_KEY", "PINECONE_API_KEY"):
        os.environ[name] = userdata.get(name)
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

faltan = [n for n in ("GOOGLE_API_KEY", "PINECONE_API_KEY") if not os.getenv(n)]
print("Faltan claves:", faltan or "ninguna")

## 2. Topología del grafo

El Supervisor está en el centro: cada especialista **siempre devuelve el control** al Supervisor,
y la arista condicional (`route`, anotada con `Literal`) decide el próximo nodo.

In [ ]:
from src.bootstrap import build_real_graph
from src.runner import run_query, format_trace

graph = build_real_graph()          # se arma UNA vez: carga embeddings, Pinecone y el LLM
print(graph.get_graph().draw_mermaid())

In [ ]:
# Render opcional del diagrama (usa mermaid.ink, necesita internet). Si falla, queda el código de arriba.
from IPython.display import Image, display
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("No se pudo renderizar el PNG:", type(exc).__name__)

## 3. Consulta 1 · camino completo (investigar → calcular → sintetizar)

La pregunta pide un **monto**, así que el Supervisor debe delegar primero en el Investigador
(para encontrar el artículo de la multa por mora) y después en el Analista (para calcularla).

In [ ]:
from src.runner import save_trace

q1 = "Pagué con 100 días de atraso un tributo de $50.000. ¿Qué multa por mora corresponde y de cuánto sería?"
s1 = await run_query(graph, q1)

print(format_trace(s1))
print("\nDecisiones del supervisor:", s1["step_count"], "| errores:", s1["errors"] or "ninguno")
print("\n--- RESPUESTA FINAL ---\n", s1["final_answer"])
save_trace(s1, "traza_demo_1", origin="real");

In [ ]:
# ¿Quién aportó qué? (el estado compartido registra cada contribución)
for c in s1["contributions"]:
    print(f"[{c['agent']:<8}] status={c['status']} · herramientas={c['tools_used']} · artículos={c['articles']} · {c['ms']:.0f} ms")

## 4. Consulta 2 · solo investigar

No hay nada que calcular: el Supervisor **no** debería invocar al Analista.

In [ ]:
q2 = "¿En cuántos años prescribe el derecho al cobro de los tributos y cuándo se amplía ese plazo?"
s2 = await run_query(graph, q2)

print(format_trace(s2))
print("\nAgentes que intervinieron:", [c["agent"] for c in s2["contributions"]])
print("\n--- RESPUESTA FINAL ---\n", s2["final_answer"])
save_trace(s2, "traza_demo_2", origin="real");

## 5. Consulta 3 · fuera de tema (caso de control)

No hay evidencia en el Código Tributario. Lo correcto es **decirlo** en vez de inventar una receta
o un artículo.

In [ ]:
q3 = "¿Cuál es la receta tradicional del chivito uruguayo?"
s3 = await run_query(graph, q3)

print(format_trace(s3))
print("\n--- RESPUESTA FINAL ---\n", s3["final_answer"])
save_trace(s3, "traza_demo_3", origin="real");

## 6. Resumen de la delegación

Una fila por consulta: qué agentes intervinieron y cuántas decisiones tomó el Supervisor
(siempre acotadas por `MAX_STEPS`).

In [ ]:
from src import config

print(f"{'consulta':<10} {'agentes':<22} {'decisiones':>10}  {'errores':>7}")
for i, s in enumerate((s1, s2, s3), 1):
    agentes = " → ".join(c["agent"] for c in s["contributions"]) or "(ninguno)"
    print(f"{i:<10} {agentes:<22} {s['step_count']:>10}  {len(s['errors']):>7}")
print(f"\nTecho de pasos configurado: MAX_STEPS={config.MAX_STEPS}, MAX_CALLS_PER_AGENT={config.MAX_CALLS_PER_AGENT}")

## 7. Aislamiento de contexto

Cada especialista recibe **solo** la pregunta original y la instrucción del Supervisor
(no el historial ni el prompt del Supervisor). Esta instrucción es lo único que "viaja":

In [ ]:
for c in s1["contributions"]:
    print(f"[{c['agent']}] instrucción recibida del Supervisor:\n   {c['instruction']}\n")

---
## Modo simulación (sin API keys)

Corre el **mismo grafo, el mismo Supervisor y las mismas herramientas**, pero con un LLM guionado y
un retriever léxico local. Sirve para ver el flujo de delegación sin gastar tokens. Las respuestas
quedan marcadas con `[SIMULACIÓN]`.

In [ ]:
from src.simulation import SIM_QUESTION, build_simulation_graph

sim_graph, *_ = build_simulation_graph()
sim = await run_query(sim_graph, SIM_QUESTION)
print(format_trace(sim))
print("\n", sim["final_answer"])